<a href="https://colab.research.google.com/github/hanidew/WIE3007-DMW-GroupProject/blob/main/DMW_GA2_Generate_Synthetic_Data_(Cleaned).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Simulation & Feature Engineering

In [1]:
# ================================
# STEP 1: Install dependencies (UPDATED)
# ================================
!pip install -q torch torchvision torchaudio transformers accelerate bitsandbytes huggingface_hub pandas > /dev/null
!pip install -q ctgan  # <--- NEW: Install CTGAN
!pip install -q -U bitsandbytes # <--- FIX: Ensure latest bitsandbytes
print("✅ Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.3 MB/s eta 0:00:00
✅ Dependencies installed.


In [2]:
# ================================
# STEP 2: Secure Login (FIXED)
# ================================
import os
from huggingface_hub import login
from google.colab import userdata

try:
    # This grabs the token from the "Key" icon on the left sidebar
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Logged in securely.")
except Exception as e:
    print("❌ Error: Could not find HF_TOKEN. Make sure you added it to the Secrets tab (Key icon)!")

# ================================
# STEP 3: Load Model (4-bit)
# ================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# NEW:
model_name = "microsoft/Phi-3-mini-4k-instruct"

print("⏳ Loading model... (this takes 2-3 minutes)")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)
print("✅ Model loaded.")

# ================================
# STEP 4: Prompt builder
# ================================
def build_prompt(n):
    return f"""[INST] Act as a Senior Risk Analyst and Synthetic Data Engine. Generate exactly {n} rows of realistic loan data.

FORMAT RULES (STRICT):
- One record per line
- Use | as delimiter
- No commas anywhere (especially in numbers like Annual_Income and Loan_Amount).
- No header row
- No explanations
- NO intro text like "Here are the records"
- NO conversational text.
- Raw text ONLY. Do NOT use markdown code blocks.

FIELDS ORDER:
Customer_ID|Age|Annual_Income|Credit_Score|Loan_Amount|Loan_Term_Months|Loan_Officer_Note|Default_Status

CONSTRAINTS (CRITICAL):
- Customer_ID: Format CUST-XXXX (e.g., CUST-0023), basically the format is like "CUST-" followed by 4 random digits.
- Age: 18–80
- Annual_Income: 20000–300000 (Integer only, NO commas, NEVER 0.)
- Credit_Score: 300–850 (Integer only)
- Loan_Amount: 5000–150000 (Integer only, NO commas)
- Loan_Term_Months: Choose standard terms (randomly choose from this selection): 12, 24, 36, 48, 60, or 72
- Default_Status: 0 (Good) or 1 (Default). Target 20% are 1.
- Loan_Officer_Note: 8-20 words. STRICTLY NO delimiter '|' inside.

LOGIC MAPPING (STRICT RELATIONSHIPS):
1. IF STATUS = 1 (DEFAULT):
   - Credit_Score MUST be between 300 and 600.
   - Loan_Officer_Note MUST use a Negative Trigger (e.g., "gambling", "lawsuit").
2. IF STATUS = 0 (GOOD):
   - Credit_Score MUST be between 650 and 850.
   - Loan_Officer_Note MUST use a Positive Trigger (e.g., "inheritance", "bonus") OR a Generic phrase.

NOTE GENERATION (Critical for Feature Extraction):
- Mix exactly ONE "Segment Hint" with ONE "Risk Context" per note.
- DO NOT start every sentence with "Client" or "Applicant". Vary the structure.
- STRICT RULE: Never use the words 'Default', 'Status', '1', or '0' inside the Loan_Officer_Note. Describe the behavior (e.g., 'unstable income') but never the result.

- SEGMENT HINTS (You MUST use one of these exact keywords to ensure classification and VARY these often):
   - Medical: "doctor", "nurse", "dentist", "pharmacist", "surgeon", "clinic"
   - Tech: "software engineer", "IT specialist", "developer", "cybersecurity", "programmer"
   - Education: "teacher", "professor", "tutor", "academic", "university"
   - Trade: "plumber", "mechanic", "electrician", "construction", "factory worker", "blue collar"
   - Service: "chef", "waiter", "artist", "designer", "retail", "driver"
   - Business: "manager", "consultant", "executive", "accountant", "sales rep", "broker"
   - Retired: "retiree", "pensioner", "elder"
   - Student: "student", "intern", "graduate"

- RISK CONTEXT (Must align with Status):
  - STATUS 1 (Default - Negative Triggers): "gambling losses", "lawsuit pending", "medical emergency costs", "unstable contracts", "over-leveraged assets", "declining revenue", "divorce settlement cost".
  - STATUS 0 (Good - POSITIVE ASSET TRIGGERS): You MUST randomly include one of these exact words for strong candidates: "inheritance", "bonus", "venture capital", "trust fund", "dividend", "royalty", "insurance settlement", "windfall".
  - STATUS 0 (Good - General): "steady cashflow", "strong savings", "low DTI", "early promotion".

ANOMALY & NOISE INSTRUCTIONS (CRITICAL FOR REALISM):
To challenge the ML model, 7% of these {n} rows must be 'Edge Cases':
1. THE UNEXPECTED DEFAULT: High Income (>100k) and High Credit (>750) but Status=1.
   Note must justify this (e.g., "Sudden medical emergency" or "Legal settlement").
2. THE UNEXPECTED GOOD PAYER: Low Income (<30k) and Low Credit (<550) but Status=0.
   Note must justify this (e.g., "Strong family guarantor" or "Asset-backed loan").
3. THE BORDERLINE CASE: Loan_Amount is exactly 2.5x Income. Make some 0 and some 1.

STRICT NUMERICAL CONSISTENCY:
Except for the 7% anomalies, ensure your logic is followed perfectly.

TWO PERFECT EXAMPLES (Follow this format):
CUST-9102|68|45000|710|120000|60|Client relies on fixed pension but loan amount is dangerously high for retirement budget|1
CUST-3321|29|92000|740|25000|24|Applicant has steady cashflow from tech stack job and low debt ratio|0

Generate {n} rows now. [/INST]
"""


✅ Logged in securely.
⏳ Loading model... (this takes 2-3 minutes)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cuda:0


✅ Model loaded.


In [4]:
# ================================
# STEP 5: Full Data Generation
# ================================
import csv
from tqdm import tqdm
import pandas as pd
import numpy as np

TARGET_ROWS = 1200
BATCH_SIZE = 50

header = [
    "Customer_ID", "Age", "Annual_Income", "Credit_Score",
    "Loan_Amount", "Loan_Term_Months", "Loan_Officer_Note",
    "Default_Status"
]

rows = []

print(f"🚀 Starting generation of {TARGET_ROWS} rows...")

# Initialize progress bar
pbar = tqdm(total=TARGET_ROWS, desc="Generating Rows")

while len(rows) < TARGET_ROWS:
    # 1. Ask Model for Data
    prompt = build_prompt(BATCH_SIZE)

    try:
        output = generator(
            prompt,
            max_new_tokens=2048,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            return_full_text=False
        )

        # 2. Process Output
        lines = output[0]["generated_text"].strip().split("\n")

        for line in lines:
            values = [v.strip() for v in line.split("|")]

            # Validation: Must have exactly 8 columns
            if len(values) == len(header):
                rows.append(values)
                pbar.update(1)

                if len(rows) >= TARGET_ROWS:
                    break

    except Exception as e:
        print(f"⚠️ Minor error in batch: {e}")
        continue

pbar.close()
print(f"\n✅ Generation Complete! Total rows in memory: {len(rows)}")

# ================================
# STEP 6: SMART LOGIC ENFORCEMENT (Fixed & Connected)
# ================================
def enforce_business_logic(df):
    print("🔧 Running Python Logic Enforcement (With Gaussian Realism & Signal Booster)...")

    # Helper for realistic Gaussian numbers (clipped to bounds)
    def get_gaussian_scores(mean, std, low, high, size):
        vals = np.random.normal(mean, std, size)
        return np.clip(vals, low, high).astype(int)

    # ---------------------------------------------------------
    # 1. SANITIZE BASICS (Fix LLM Hallucinations)
    # ---------------------------------------------------------
    # Identify rows with invalid scores (0, or >850, or <300)
    mask_bad_score = (df['Credit_Score'] < 300) | (df['Credit_Score'] > 850)

    # Repair Defaults -> Gaussian around 520 (Bell curve)
    count_bad_def = (mask_bad_score & (df['Default_Status'] == 1)).sum()
    if count_bad_def > 0:
        df.loc[mask_bad_score & (df['Default_Status'] == 1), 'Credit_Score'] = \
            get_gaussian_scores(520, 50, 350, 580, count_bad_def)

    # Repair Good -> Gaussian around 720
    count_bad_good = (mask_bad_score & (df['Default_Status'] == 0)).sum()
    if count_bad_good > 0:
        df.loc[mask_bad_score & (df['Default_Status'] == 0), 'Credit_Score'] = \
            get_gaussian_scores(720, 50, 650, 820, count_bad_good)

    # Ensure Income is never 0 or too low (Gaussian around 45k for low end)
    mask_low_inc = df['Annual_Income'] < 10000
    if mask_low_inc.sum() > 0:
        df.loc[mask_low_inc, 'Annual_Income'] = \
            get_gaussian_scores(45000, 5000, 30000, 60000, mask_low_inc.sum())

    # ---------------------------------------------------------
    # 2. ENFORCE "ANOMALY" LOGIC (Varied Text Injection)
    # ---------------------------------------------------------

    # --- Anomaly A: Rich Defaulters ---
    candidates_rich = df[(df['Annual_Income'] > 100000) & (df['Credit_Score'] > 720)]
    if len(candidates_rich) > 0:
        # Target 20%
        idx_rich = candidates_rich.sample(frac=0.30, random_state=42).index
        df.loc[idx_rich, 'Default_Status'] = 1

        reasons = [
            " [Risk: Large gambling debt]", " [Risk: Pending lawsuit]",
            " [Risk: Speculative losses]", " [Risk: Hidden liabilities]",
            " [Risk: Divorce settlement]", " [Risk: Casino addiction]"
        ]
        random_reasons = np.random.choice(reasons, size=len(idx_rich))
        df.loc[idx_rich, 'Loan_Officer_Note'] = df.loc[idx_rich, 'Loan_Officer_Note'] + random_reasons

    # --- Anomaly B: Poor Good Payers ---
    candidates_poor = df[(df['Annual_Income'] < 40000) & (df['Credit_Score'] < 580)]
    if len(candidates_poor) > 0:
        idx_poor = candidates_poor.sample(frac=0.30, random_state=42).index
        df.loc[idx_poor, 'Default_Status'] = 0

        assets = [
            " [Asset: Family Guarantor]", " [Asset: Inheritance Trust]",
            " [Asset: Locked Savings]", " [Asset: Co-signer Verified]",
            " [Asset: Insurance Payout]"
        ]
        random_assets = np.random.choice(assets, size=len(idx_poor))
        df.loc[idx_poor, 'Loan_Officer_Note'] = df.loc[idx_poor, 'Loan_Officer_Note'] + random_assets

    # ---------------------------------------------------------
    # 3. ENFORCE "BORDERLINE" LOGIC (Math Check)
    # ---------------------------------------------------------
    # "Standard" Over-leveraged people (High DTI) should default
    df['DTI_Ratio'] = df['Loan_Amount'] / df['Annual_Income']

    # Filter: High DTI + Good Status + NO Asset keywords
    mask_risk = (df['DTI_Ratio'] > 2.5) & \
                (df['Default_Status'] == 0) & \
                (~df['Loan_Officer_Note'].str.contains("Guarantor|Inheritance|Asset|Co-signer", case=False, na=False))

    df.loc[mask_risk, 'Default_Status'] = 1
    df.drop(columns=['DTI_Ratio'], inplace=True)

    # ---------------------------------------------------------
    # 4. SIGNAL BOOSTER (90% Rule)
    # ---------------------------------------------------------
    # Find Defaulters who MIGHT be missing a negative keyword
    risk_keywords = "gambling|lawsuit|medical|unstable|over-leveraged|declining|divorce|risk|debt"
    weak_defaulters = (df['Default_Status'] == 1) & (~df['Loan_Officer_Note'].str.contains(risk_keywords, case=False))

    # Force inject risk phrase into MOST (but not all) weak defaulters
    if weak_defaulters.sum() > 0:
        weak_indices = df[weak_defaulters].index

        # Select 90% to fix (Leave 10% as "Mystery Defaults" to challenge the model)
        # This prevents the model from relying ONLY on text
        indices_to_fix = np.random.choice(weak_indices, size=int(len(weak_indices) * 0.7), replace=False)

        extra_risks = [
            " [Note: History of late payments]",
            " [Note: High debt-to-income]",
            " [Note: erratic cash flow]"
        ]

        if len(indices_to_fix) > 0:
            df.loc[indices_to_fix, 'Loan_Officer_Note'] = df.loc[indices_to_fix, 'Loan_Officer_Note'] + np.random.choice(extra_risks, size=len(indices_to_fix))
            print(f"✅ Signal Booster applied to {len(indices_to_fix)} rows (leaving some mysteries).")

    print("✅ Logic Enforcement Complete. Data is clean, varied, and realistic.")
    return df

# ================================
# THE BRIDGE: Prepare Data & Apply Logic
# ================================
# 1. Create Raw DataFrame from list
df_raw = pd.DataFrame(rows[:TARGET_ROWS], columns=header)

# 2. Convert Columns to Numeric (Essential for the logic to work!)
num_cols = ["Age", "Annual_Income", "Credit_Score", "Loan_Amount", "Loan_Term_Months", "Default_Status"]
for col in num_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# 3. Drop rows where numeric conversion failed (NaNs)
df_raw.dropna(subset=num_cols, inplace=True)

# 4. CALL THE FUNCTION (The critical step!)
df_final = enforce_business_logic(df_raw)

# ================================
# STEP 7: Save RAW Dataset
# ================================
output_file = "synthetic_financial_data_FINAL.csv"
df_final.to_csv(output_file, sep='|', index=False, header=True)
print(f"\n💾 File saved successfully: {output_file}")

# ================================
# STEP 8: Verify Dataset
# ================================
try:
    df_verify = pd.read_csv(output_file, sep='|')
    print("\n📊 DATASET SUMMARY:")
    print(f"Shape: {df_verify.shape}")
    print("-" * 30)
    print(df_verify.head())
    print("-" * 30)
    print("\n🔍 Checking Class Balance:")
    print(df_verify['Default_Status'].value_counts(normalize=True))

    print("\n🔍 Checking Score Integrity (Should be 300-850):")
    print(f"Min: {df_verify['Credit_Score'].min()}, Max: {df_verify['Credit_Score'].max()}")

    # Quick Correlation Check
    corr = df_verify['Credit_Score'].corr(df_verify['Default_Status'])
    print(f"\n🔍 Correlation (Score vs Default): {corr:.4f} (Should be negative)")

except Exception as e:
    print("❌ Error reading CSV:", e)

🚀 Starting generation of 1200 rows...



Generating Rows:   6%|▌         | 69/1200 [44:09<12:03:43, 38.39s/it]

Generating Rows:   4%|▍         | 51/1200 [06:06<2:45:04,  8.62s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset

Generating Rows: 100%|██████████| 1200/1200 [1:36:17<00:00,  4.81s/it]



✅ Generation Complete! Total rows in memory: 1200
🔧 Running Python Logic Enforcement (With Gaussian Realism & Signal Booster)...
✅ Signal Booster applied to 217 rows (leaving some mysteries).
✅ Logic Enforcement Complete. Data is clean, varied, and realistic.

💾 File saved successfully: synthetic_financial_data_FINAL.csv

📊 DATASET SUMMARY:
Shape: (1091, 8)
------------------------------
  Customer_ID   Age  Annual_Income  Credit_Score  Loan_Amount  \
0   CUST-4321  55.0       120000.0         770.0     160000.0   
1   CUST-2056  40.0       150000.0         780.0      90000.0   
2   CUST-9812  34.0        75000.0         660.0     110000.0   
3   CUST-8132  28.0        32000.0         710.0      60000.0   
4   CUST-0543  45.0        80000.0         820.0     240000.0   

   Loan_Term_Months                                  Loan_Officer_Note  \
0              48.0  Teaching assistant facing sudden hospital bill...   
1              36.0  Photographer inherits art collection easing fi..

# Clean Dataset

In [5]:
import pandas as pd

# 1. Load the raw file
df = pd.read_csv('synthetic_financial_data_FINAL.csv', sep='|')

# 2. Fix the Customer IDs (Standardize to 'CUST-XXXX')
# This replaces any underscore '_' with a hyphen '-'
df['Customer_ID'] = df['Customer_ID'].str.replace('_', '-', regex=False)

# 3. Convert Floats to Integers
# We explicitly convert these columns to 'int64' to remove the ".0"
cols_to_fix = ["Age", "Annual_Income", "Credit_Score", "Loan_Amount", "Loan_Term_Months", "Default_Status"]
for col in cols_to_fix:
    df[col] = df[col].astype(int)

# 4. Save the "Gold Standard" version
df.to_csv('synthetic_financial_data_CLEAN.csv', sep='|', index=False)

print("✨ Dataset Polished!")
print(df.head())
print(f"\nVerifying IDs: {df['Customer_ID'].str.contains('_').sum()} underscores remaining (Should be 0).")

✨ Dataset Polished!
  Customer_ID  Age  Annual_Income  Credit_Score  Loan_Amount  \
0   CUST-4321   55         120000           770       160000   
1   CUST-2056   40         150000           780        90000   
2   CUST-9812   34          75000           660       110000   
3   CUST-8132   28          32000           710        60000   
4   CUST-0543   45          80000           820       240000   

   Loan_Term_Months                                  Loan_Officer_Note  \
0                48  Teaching assistant facing sudden hospital bill...   
1                36  Photographer inherits art collection easing fi...   
2                48  Retail store owner deals daily challenges due ...   
3                24  Software developer secures bonus which helps c...   
4                72  Business consultant recently benefited from su...   

   Default_Status  
0               1  
1               0  
2               1  
3               0  
4               1  

Verifying IDs: 0 underscores 

# Feature Engineering

CPU version

In [7]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from google.colab import drive

# ---------------------------------------------------------
# 0. SETUP
# ---------------------------------------------------------
# drive.mount('/content/drive')  # Uncomment if using Drive
file_path = 'synthetic_financial_data_CLEAN.csv'  # Using the clean version

try:
    df = pd.read_csv(file_path, sep='|')
    print(f"✅ Loaded {len(df)} rows.")
except FileNotFoundError:
    print("❌ File not found. Make sure you ran the 'Cleaning' step!")

# ---------------------------------------------------------
# 1. DEFINE EXTRACTION LOGIC (Matched to Generator)
# ---------------------------------------------------------
def extract_features(row):
    note = str(row['Loan_Officer_Note']).lower()

    # --- A. OCCUPATION (Matched to Prompt Keywords) ---
    # We refined these to match the specific words the LLM was told to use
    # --- UPDATED OCCUPATION LOGIC (Catches ~90% of jobs) ---
    jobs = {
        'Medical':   r'\b(doctor|nurse|dentist|pharmacist|surgeon|clinic|practitioner|medical)\b',
        'Tech':      r'\b(software|developer|it specialist|cybersecurity|programmer|tech|data|engineer)\b',
        'Education': r'\b(teacher|professor|tutor|academic|university|teaching|faculty)\b',
        'Trade':     r'\b(plumber|mechanic|electrician|construction|factory|blue collar|technician|driver)\b',
        'Service':   r'\b(chef|waiter|waitress|artist|designer|retail|clerk|staff|customer|service)\b',
        'Business':  r'\b(manager|consultant|executive|accountant|sales|broker|business|owner|professional|founder|entrepreneur)\b',
        'Real Estate': r'\b(estate|agent|realtor|property|landlord)\b',  # NEW CATEGORY
        'Retired':   r'\b(retiree|pension|elder|retired)\b',
        'Student':   r'\b(student|intern|graduate)\b'
    }

    found_job = "Other"
    for category, pattern in jobs.items():
        if re.search(pattern, note):
            found_job = category
            break

    # --- B. POSITIVE ASSETS (The "Good" Signal) ---
    # Keywords: inheritance, bonus, venture, trust, dividend, royalty, insurance, windfall
    asset_pattern = r'\b(inheritance|bonus|venture|trust|dividend|royalty|insurance|windfall)\b'
    has_asset = 1 if re.search(asset_pattern, note) else 0

    # --- C. NEGATIVE RISKS (The "Bad" Signal - NEW!) ---
    # Keywords: gambling, lawsuit, medical emergency, unstable, over-leveraged, divorce, late payment
    # This is crucial because of the "Signal Booster" we ran earlier.
    risk_pattern = r'\b(gambling|lawsuit|medical|unstable|over-leveraged|declining|divorce|late payment|erratic|debt)\b'
    risk_flag = 1 if re.search(risk_pattern, note) else 0

    return pd.Series([found_job, has_asset, risk_flag])

# ---------------------------------------------------------
# 2. RUN EXTRACTION
# ---------------------------------------------------------
print("🚀 Running Smart Feature Extraction...")
tqdm.pandas()

# Apply the text mining
df[['Occupation', 'Has_Asset', 'Risk_Flag']] = df.progress_apply(extract_features, axis=1)

# ---------------------------------------------------------
# 3. ADD MATHEMATICAL FEATURES (The "DTI" Rule)
# ---------------------------------------------------------
print("🧮 Calculating Financial Ratios...")

# DTI (Debt-to-Income): This captures the "Borderline" logic we enforced
# We add +1 to Income to avoid division by zero (just in case)
df['DTI_Ratio'] = df['Loan_Amount'] / (df['Annual_Income'] + 1)

# Log Income (Standard ML practice to normalize high incomes)
df['Log_Income'] = np.log1p(df['Annual_Income'])

# ---------------------------------------------------------
# 4. SAVE & PREVIEW
# ---------------------------------------------------------
output_file = 'synthetic_data_cleaned_feature_engineered.csv'
df.to_csv(output_file, sep='|', index=False)

print(f"\n✅ Success! Engineered data saved to: {output_file}")
print("-" * 50)
print(df[['Loan_Officer_Note', 'Occupation', 'Risk_Flag', 'DTI_Ratio']].head())
print("-" * 50)

# Check correlation of new features
print("\n🔍 Correlation with Default:")
print(df[['Risk_Flag', 'Has_Asset', 'DTI_Ratio', 'Credit_Score', 'Default_Status']].corr()['Default_Status'])

✅ Loaded 1091 rows.
🚀 Running Smart Feature Extraction...


100%|██████████| 1091/1091 [00:00<00:00, 8175.20it/s] 

🧮 Calculating Financial Ratios...

✅ Success! Engineered data saved to: synthetic_data_cleaned_feature_engineered.csv
--------------------------------------------------
                                   Loan_Officer_Note Occupation  Risk_Flag  \
0  Teaching assistant facing sudden hospital bill...  Education          0   
1  Photographer inherits art collection easing fi...      Other          0   
2  Retail store owner deals daily challenges due ...    Service          0   
3  Software developer secures bonus which helps c...       Tech          0   
4  Business consultant recently benefited from su...   Business          0   

   DTI_Ratio  
0   1.333322  
1   0.599996  
2   1.466647  
3   1.874941  
4   2.999963  
--------------------------------------------------

🔍 Correlation with Default:
Risk_Flag         0.578122
Has_Asset        -0.137505
DTI_Ratio         0.451912
Credit_Score      0.028142
Default_Status    1.000000
Name: Default_Status, dtype: float64
